In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://enhantica.github.io/edi-download/pypi/simple/


# Sequential fit: LATP, BM01

Fit three phases, LATP and two AlPO₄ forms, to 73 synchrotron X-ray patterns from BM01 at ESRF,
one after another, with linked displacement parameters.

The project is the catalogue case `pd-xray-cwl_latp-bm01_scan`, the one the command line and the
app's Examples open. Al shares Ti's site, coordinate and displacement, and the impurity phase's
displacements are shared, from the start.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Sequential</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">xray</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import csv
from pathlib import Path

import easydiffraction as edi

## Load the project

A sequential fit writes one row per pattern into the project's `analysis/results.csv`, so work
on a copy of the project, saved to a folder of its own.

In [3]:
project_path = edi.download_data('pd-xray-cwl_latp-bm01_scan', destination='data')
project = edi.Project.load(project_path)
project.save_as('pd-xray-cwl_latp-bm01_scan')
latp = project.structures['latp']
c2221 = project.structures['alpo4_c2221']
p63mc = project.structures['alpo4_p63mc']
experiment = project.experiments['bm01']
project.display.constraints()

Data 'pd-xray-cwl_latp-bm01_scan' is at data/pd-xray-cwl_latp-bm01_scan-67292a68d53c/pd-xray-cwl_latp-bm01_scan
Saving project to 'pd-xray-cwl_latp-bm01_scan'


View(html=<102,844 characters>, height=460)

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

## Stage the template fit

The template is the scan's first pattern. Fit it alone first, in stages, so the scan starts from
a good model: switch the fitting mode to a single fit.

In [5]:
project.analysis.fitting_mode = 'single'

In [6]:
project.display.pattern('bm01')

View(html=<651,693 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scales, 2θ zero and cells

The three phases' scales and cells and the zero shift.

In [7]:
experiment.linked_structures['latp'].scale.free = True
experiment.linked_structures['alpo4_c2221'].scale.free = True
experiment.linked_structures['alpo4_p63mc'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
latp.cell.length_a.free = True
latp.cell.length_c.free = True
c2221.cell.length_a.free = True
c2221.cell.length_b.free = True
c2221.cell.length_c.free = True
p63mc.cell.length_a.free = True
p63mc.cell.length_c.free = True

In [8]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,alpo4_c2221,cell,,length_a,6.97,0.03,0,30,Å
2,alpo4_c2221,cell,,length_b,7.11,0.03,0,30,Å
3,alpo4_c2221,cell,,length_c,7.004,0.019,0,30,Å
4,alpo4_p63mc,cell,,length_a,4.983,0.007,0,30,Å
5,alpo4_p63mc,cell,,length_c,9.504,0.019,0,30,Å
6,latp,cell,,length_a,8.4953,0.0003,0,30,Å
7,latp,cell,,length_c,20.8268,0.0009,0,30,Å
8,bm01,instrument,,calib_twotheta_offset,-0.0388,0.0014,-inf,inf,deg
9,bm01,linked_structure,latp,scale,4.15,0.04,-inf,inf,
10,bm01,linked_structure,alpo4_c2221,scale,0.54,0.12,-inf,inf,


In [9]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-xray-cwl_latp-bm01_scan, 3114 pts fitted / 3163 loaded, 11 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.51 140198364.17
      1      0.023      4.51 140142449.16   
      2      0.074      4.51 140112444.85   
      3      0.098      4.51 140108484.48   
      4      0.125      4.51 140107944.41   
      5      0.158      4.51 140107929.77   
  done · converged · 5 iters · reduced χ² 140107929.77 · Rwp 4.51 % · 0.18 s


View(html=<94,041 characters>, height=376)

In [10]:
project.display.pattern('bm01')

View(html=<652,115 characters>, height=480)

### Background

In [11]:
for point in experiment.background:
    point.intensity.free = True

In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-xray-cwl_latp-bm01_scan, 3114 pts fitted / 3163 loaded, 41 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.51 141475726.02
      1      0.092      4.51 141465714.66   
      2      0.147      4.51 141465291.95   
      3      0.229      4.51 141465280.04   
  done · converged · 3 iters · reduced χ² 141465280.04 · Rwp 4.51 % · 0.25 s


View(html=<126,923 characters>, height=376)

In [13]:
project.display.pattern('bm01')

View(html=<652,755 characters>, height=480)

### Peak widths

In [14]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.mixing_eta_0.free = True

In [15]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-xray-cwl_latp-bm01_scan, 3114 pts fitted / 3163 loaded, 45 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.51 141649659.68
      1      0.102      4.51 141617725.69   
      2      0.167      4.51 141616397.51   
      3      0.195      4.51 141616173.71   
      4      0.229      4.51 141612847.55   
      5      0.339      4.51 141612757.46   
  done · converged · 5 iters · reduced χ² 141612757.46 · Rwp 4.51 % · 0.37 s


View(html=<131,242 characters>, height=376)

In [16]:
project.display.pattern('bm01')

View(html=<652,728 characters>, height=480)

### Peak asymmetry

In [17]:
experiment.peak.asym_beba_a0.free = True
experiment.peak.asym_beba_b0.free = True

In [18]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-xray-cwl_latp-bm01_scan, 3114 pts fitted / 3163 loaded, 47 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.51 141705103.30
      1      0.096      4.51 141704033.35   
      2      0.159      4.51 141697567.66   
      3      0.262      4.51 141697436.83   
  done · converged · 3 iters · reduced χ² 141697436.83 · Rwp 4.51 % · 0.29 s


View(html=<133,299 characters>, height=376)

In [19]:
project.display.pattern('bm01')

View(html=<652,807 characters>, height=480)

### LATP positions

The main phase's free coordinates; Al's follows Ti's.

In [20]:
latp.atom_sites['Ti1'].fract_z.free = True
latp.atom_sites['P1'].fract_x.free = True
latp.atom_sites['O1'].fract_x.free = True
latp.atom_sites['O1'].fract_y.free = True
latp.atom_sites['O1'].fract_z.free = True
latp.atom_sites['O2'].fract_x.free = True
latp.atom_sites['O2'].fract_y.free = True
latp.atom_sites['O2'].fract_z.free = True

In [21]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-xray-cwl_latp-bm01_scan, 3114 pts fitted / 3163 loaded, 55 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.51 142068008.74
      1      0.099      4.51 142054983.72   
      2      0.163      4.51 142052860.61   
      3      0.199      4.51 142052190.88   
      4      0.238      4.51 142050716.46   
      5      0.331      4.51 142050690.48   
  done · converged · 5 iters · reduced χ² 142050690.48 · Rwp 4.51 % · 0.37 s


View(html=<141,907 characters>, height=376)

In [22]:
project.display.pattern('bm01')

View(html=<652,839 characters>, height=480)

### LATP displacements

In [23]:
latp.atom_sites['Ti1'].adp_iso.free = True
latp.atom_sites['P1'].adp_iso.free = True
latp.atom_sites['O1'].adp_iso.free = True
latp.atom_sites['O2'].adp_iso.free = True

In [24]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-xray-cwl_latp-bm01_scan, 3114 pts fitted / 3163 loaded, 59 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.51 142236681.57
      1      0.096      4.51 142234344.23   
      2      0.156      4.51 142232766.12   
      3      0.195      4.51 142229314.14   
      4      0.233      4.51 142228665.28   
      5      0.330      4.51 142228616.00   
  done · converged · 5 iters · reduced χ² 142228616.00 · Rwp 4.51 % · 0.37 s


View(html=<146,200 characters>, height=376)

In [25]:
project.display.pattern('bm01')

View(html=<652,853 characters>, height=480)

In [26]:
project.display.fit.correlations()

,1,2,3,4,5
1 bm01.peak.broad_gauss_u,,,,,
2 bm01.peak.broad_gauss_v,-0.94,,,,
3 bm01.peak.broad_gauss_w,0.74,-0.89,,,
4 bm01.peak.asym_beba_a0,-0.01,0.02,-0.02,,
5 bm01.peak.asym_beba_b0,-0.21,0.31,-0.46,0.80,


In [27]:
project.display.structure('latp')

View(html=<6,509,860 characters>, height=660)

In [28]:
project.display.structure('alpo4_c2221')

View(html=<228,288 characters>, height=660)

In [29]:
project.display.structure('alpo4_p63mc')

View(html=<142,357 characters>, height=660)

## Fit the scan

Back to the sequential mode, with the template's free parameters. Each pattern starts from the
fit before it.

In [30]:
project.analysis.fitting_mode = 'sequential'
fitted = []
result = project.fit_sequential(on_file_complete=lambda *args: fitted.append(args))
print(result.status.name.lower(), 'after', len(fitted), 'patterns')

done after 73 patterns


## The results

One row per pattern, in the order they were fitted.

In [31]:
results = Path('pd-xray-cwl_latp-bm01_scan/analysis/results.csv')
with results.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
print(len(rows), 'rows; columns:', ', '.join(list(rows[0])[:6]), '...')

73 rows; columns: file_path, fit_result.reduced_chi_square, fit_result.success, fit_result.iterations, alpo4_c2221.cell.length_a, alpo4_c2221.cell.length_a.uncertainty ...


## Save the report

In [32]:
project.report.save_html('pd-xray-cwl_latp-bm01_scan.html')

Saving report to 'pd-xray-cwl_latp-bm01_scan.html'
